In [ ]:
%pip install --upgrade pip
%pip install yfinance torch numpy scikit-learn

In [65]:
import yfinance as yf
import torch.nn as nn
import numpy as np
import torch 
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timedelta

In [ ]:
AAPL = yf.Ticker("AAPL")
AAPL.info

In [ ]:
AAPL_hist = AAPL.history(period="1mo")
AAPL_hist.info

In [68]:
ticker = "AAPL"
Seq_len = 60 #number of months for the input window
Hidden_size = 128
num_layers = 2
dropout = .2
batch_size = 32
lr = 1e-3
epochs = 20
Use_log_returns = True
Device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using: {Device}")


using: mps


In [69]:
def fetch_ohlcv(ticker: str, years: int = 5) -> np.array:
    end = datetime.today()
    start = end - timedelta(days = years*365)
    df = yf.download(ticker, start, end, auto_adjust = True, progress = False)
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    return df.values.astype(np.float64), df.index.tolist()
raw, dates = fetch_ohlcv(ticker=ticker)
print(f"Fetched {len(raw)} trading days for {ticker}")

Fetched 1255 trading days for AAPL


In [70]:
def compute_features(raw: np.ndarray) ->np.ndarray:
    if Use_log_returns:
        price = raw[:,:4]
        volume = raw[:,:4:5]

        log_ret = np.log(price[1:] /  price[:-1])
        vol_chg = np.log(volume[1:] /  volume[:-1])

        close_ret = log_ret[:,3]
        rolling_std = np.array([
            close_ret[max(0, i-9): i+1].std()
            for i in range(len(close_ret))
        ]).reshape(-1, 1)

        features = np.hstack([log_ret, vol_chg, rolling_std])
    else:
        scaler = StandardScaler()
        features = scaler.fit.transform(raw)
        features = features[1:]
    return features.astype(np.float32)

features = compute_features(raw)
print(f"feature matrix shape: {features.shape} (timesteps, features)")


feature matrix shape: (1254, 6) (timesteps, features)


In [74]:
class priceSequenceDataset(Dataset):
    def __init__(self, features: np.ndarray, seq_len: int = Seq_len):
        self.features = torch.tensor(features)
        self.seq_len = seq_len
    
    def __len__(self):
        return len(self.features) - self.seq_len
    
    def __getitem__(self,idx):
        x = self.features[idx: idx + self.seq_len]
        y = self.features[idx + self.seq_len, 3]
        return x,y
    
dataset = priceSequenceDataset(features)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last = True)
print(f"Dataset: {len(dataset)} samples | Batches per epoch {len(loader)}")

Dataset: 1194 samples | Batches per epoch 37
